In [ ]:
"""
Important Note:
the code has the file directory setup in google drive. Executing the same code in local system will create files in the local directory with the folder names as specified in the code.
You can also observe the results of each cell.
"""

In [ ]:
"""
import required libraries
"""
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from itertools import product
from ast import literal_eval
import json

In [ ]:
"""
:function: prepare_duplicate_instances_data: This function creates a duplicate instance dataset which will be used to 
train the contextual bandit (CB) model for all possible actions for a given context.

:params: a pandas dataframe representing the final embedded dataset and the embedding model name such as gpt2, bert, word2vec

:return: duplicate instance dataset (action_context_map) with each action mapped to each context for the input 10,000 data instances.

:info: The size of input dataset is 250,000 data instance. Out of which only 10,000 instances are selected for this expriement. 
The number of unique actions are 19. 
Hence the size of the duplicate instance dataset is 19 * 10,000 = 190,000.
These 190,000 instances represent each possible action mapped to each possible context in the dataset. 
This dataset is created for each model. ************
"""
def prepare_duplicate_instances_data(movies_keywords_genres, model):
  movies_keywords_genres_top10k = movies_keywords_genres.head(10000)
  movies_keywords_genres_combine = movies_keywords_genres_top10k.copy()
  genres_mappings = dict()
  txt_file_path = f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/input/genre_mapping.txt'
  with open(txt_file_path, 'r') as txt_file:
      genres_mappings = json.loads(txt_file.read())
  print('Genre Mapping:', genres_mappings)
  unique_genres = list(genres_mappings.values())
  combinations = list(product(unique_genres, movies_keywords_genres_combine['keywords_embedding_string']))
  action_context_map = pd.DataFrame(combinations, columns=['genres_index', 'keywords_embedding_string'])
  action_context_map['genres_indices'] = action_context_map['keywords_embedding_string'].map(movies_keywords_genres_combine.set_index('keywords_embedding_string')['genres_indices'].to_dict())
  action_context_map['id'] = action_context_map['keywords_embedding_string'].map(movies_keywords_genres_combine.set_index('keywords_embedding_string')['id'].to_dict())
  action_context_map['original_title_embedding_string'] = action_context_map['keywords_embedding_string'].map(movies_keywords_genres_combine.set_index('keywords_embedding_string')['original_title_embedding_string'].to_dict())
  action_context_map['overview_embedding_string'] = action_context_map['keywords_embedding_string'].map(movies_keywords_genres_combine.set_index('keywords_embedding_string')['overview_embedding_string'].to_dict())
  action_context_map['release_date'] = action_context_map['keywords_embedding_string'].map(movies_keywords_genres_combine.set_index('keywords_embedding_string')['release_date'].to_dict())

  action_context_map['genres_indices'] = action_context_map['genres_indices'].apply(literal_eval)
  action_context_map.to_csv(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/data-prepare/{model}_action_context_map.csv', index=False)
  return action_context_map

In [ ]:
"""
:function: This function creates the test and train dataset from the embedded dataset.

:params: the duplicate instances dataset and the embedding model name such as gpt2, bert, word2vec

:return: test and train dataset for every model.

:info: the duplicate instance dataset created will be used to as a train and test dataset for the model. The train dataset is 80% of the duplicate instance dataset 
i.e., 152,000 data instances. 
The test dataset consists of some instances of the train data and data from the 180,000 data instances which were in the final embedded dataset but not considered while creating 
the duplicate instance dataset. The reason for creating the test dataset in such as way was to ensure that the expriemnt will be carried out on instances for which:
1. the model has some previous knowledge base about action, context and its associated rewards, i.e., test data taken from train data
2. and the data instances for which the model has no knowledge of the action, context and its associated rewards, i.e., the data instances from 
the remaining 180,000 data instances which were not used for training.

These dataset is created for each model. ************
"""
def prepare_test_and_train_data(action_context_reward_map, model):
  train_movies_df, test_movies_df = train_test_split(action_context_reward_map, test_size=0.2, random_state=42)
  print('Shape of train_movies_df', train_movies_df.shape[0])
  print('Shape of test_movies_df', test_movies_df.shape[0])
  test_movies_df.to_csv(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/data-prepare/{model}_TEST_file.csv', index=False)
  print('Data of test_movies_df', test_movies_df.head(2))
  create_train_data_df(train_movies_df)
  train_movies_df.to_csv(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/data-prepare/{model}_TRAIN_file.csv', index=False)
  print('Data of train_movies_df', train_movies_df.head(2))
  create_train_data_txt_for_vw_wabbit(train_movies_df, model)
  return train_movies_df, test_movies_df

In [ ]:
"""
:function: This function is a helper function to assign cost to train dataset.

:params: a row in dataset with attributes genres_indices (the genres the movie belongs to), the genre_index (single movie genre assigned while creating the duplicate instance
dataset)

:return: reward assigned to each row. this can range from 0 to 1.

:info: the reward is assigned to the train data for the model training using this function.

These function is executed for each model. ************
"""
def helper_train_assign_reward(row):
  try:
    index = row['genres_indices'].index(row['genres_index'])
    if index == 0:
        return 1
    elif index >= 1 :
      return 1 - (index/len(row['genres_indices']))
  except ValueError:
    return 0

In [ ]:
"""
:function: This function is a helper function to assign cost to test dataset.

:params: a row in dataset with attributes genres_indices (the genres the movie belongs to), the genre_index (single movie genre assigned while creating the duplicate instance
dataset)

:return: cost assigned to each row. this can range from 0 to 1.

:info: the cost is assigned to the train data for the model training using this function.

These function is executed for each model. ************
"""
def helper_train_assign_cost(row):
  try:
    index = row['genres_indices'].index(row['genres_index'])
    if index == 0:
        return 0
    elif index >= 1 :
      return index/len(row['genres_indices'])
  except ValueError:
    return 1

In [ ]:
"""
:function: This function is a helper function to convert the train dataset into Vowpal Wabbit input format. vowpal wabbit accepts data in a fixed format:
that is <action> : <cost> : <probability> | <features/context>.

:params: a row in dataset with attributes genres_indices (the genres the movie belongs to), the genre_index (single movie genre assigned while creating the duplicate instance
dataset) and context attributes like keywords_embedding_string, original_title_embedding_string, overview_embedding_string and other attributes like movie_id and release_date.

:return: formatted train data

:info: this function converts the train dataset 152,000 instances into the vowpal wabbit input format.

These function is executed for each model. ************
"""
def helper_train_convert_to_vw_format(row):
    keywords_embedding =  row["keywords_embedding_string"]
    original_title_embedding = row["original_title_embedding_string"]
    release_date = row["release_date"]
    overview_embedding = row["overview_embedding_string"]
    return f'{row["genres_index"]}:{row["cost"]} | feature1:{id} feature2:{keywords_embedding} feature3:{original_title_embedding} feature4:{release_date} feature5:{overview_embedding}'

In [ ]:
"""
:function: This function is a to create a train dataset by creating additional columns of cost and rewards in the earlier created train dataset.

:params: the train dataset before formatting in the vowpal wabbit format

:return: adds additional columns cost and rewards to the train dataset

:info: this function assigns the cost and rewards to the train dataset which the model will utilize for learning the relationship between various action and context and their 
associated rewards.

These function is executed for each model. ************
"""
def create_train_data_df(train_movies_df):
  train_movies_df['rewards'] = train_movies_df.apply(helper_train_assign_reward, axis=1)
  train_movies_df['cost'] = train_movies_df.apply(helper_train_assign_cost, axis=1)

In [ ]:
"""
:function: This function exports created the Vowapl Wabbit input format dataset into a .txt file.

:params: the vowpal wabbit formatted train dataset.

:return: create a .txt file for the vowpal wabbit formatted train dataset.

:info: A .txt file is created for the vowpal wabbit formatted train dataset because it is easier to parse a .txt file line by line as compared to the pandas dataframe for the
vowpal wabbit model.

These function is executed for each model. ************
"""
def create_train_data_txt_for_vw_wabbit(train_movies_df, model):
  vw_train_data = train_movies_df.apply(helper_train_convert_to_vw_format, axis=1).tolist()
  output_file_path = f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/data-prepare/{model}_vw_train_data.txt'
  with open(output_file_path, 'w') as file:
      for item in vw_train_data:
          file.write("%s\n" % item)

In [ ]:
"""
:function: This function is used to generate the second part of the test dataset as mentioned above, which is data instamces for which the model has no prior
knowledge. It randomly selects data instances from the final embedded dataset out of the remaining 180,000 instance which were not used for training the model.

:params:  a pandas dataframe representing the final embedded dataset and the embedding model name such as gpt2, bert, word2vec

:return: returns a random_test_movies_df which forms the part of the test dataset.

:info: This function randomly selects instances to create the test dataset for running iterations and making predicitions.
It uses a random state to generate the same test dataset for all the three models.

These function is executed for each model. ************

"""
def prepare_random_test_data(movies_keywords_genres, model):
   random_test_movies_df = movies_keywords_genres.sample(n=38000, random_state=2024)
   random_test_movies_df = random_test_movies_df[['keywords_embedding_string', 'genres_indices', 'id', 'original_title_embedding_string', 'release_date', 'overview_embedding_string']]
   random_test_movies_df.to_csv(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/data-prepare/{model}_RANDOM_TEST_file.csv', index=False)
   print('Data of random_test_movies_df', random_test_movies_df.head(2))
   print(random_test_movies_df.columns)
   print(random_test_movies_df.shape[0])
   return random_test_movies_df

In [ ]:
"""
:function: This is the main function to create the data required for the project.
It is implemented for all the there models gpt2, bert and word2vec to prepare the data respectively.

:params: none

:return: prepares the following files for each model:
1.	{model}_action_context_map.csv - Dataset with 10k rows. Acts as the main dataset for this project and has all attributes required for the project.
2.	{model}_TRAIN_file.csv - This is the train dataset for the model.
3.	{model}_vw_train_data.txt - This is the train dataset formatted to the Vowpal Wabbit input format.
4.	{model}_RANDOM_TEST_file.csv - This is the test dataset. It consists of data instances randomly data from the final embedded dataset as well as some instances from training data.

"""
def main_data_prepare():
  model_list = ['gpt2', 'bert', 'word2vec']
  for model in model_list:
    movies_keywords_genres = pd.read_csv(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/input/f-final_mkg_dataset_{model}.csv')
    action_context_reward_map = prepare_duplicate_instances_data(movies_keywords_genres, model)
    train_movies_df, test_movies_df = prepare_test_and_train_data(action_context_reward_map, model)
    random_test_movies_df = prepare_random_test_data(movies_keywords_genres, model)
    movies_keywords_genres = action_context_reward_map = train_movies_df = test_movies_df = random_test_movies_df = None

In [ ]:
"""
invokes the main function
"""
main_data_prepare()

Genre Mapping: {'Crime': 1, 'Western': 2, 'Fantasy': 3, 'Science Fiction': 4, 'Comedy': 5, 'Family': 6, 'Mystery': 7, 'Drama': 8, 'Animation': 9, 'Thriller': 10, 'Documentary': 11, 'Horror': 12, 'Romance': 13, 'TV Movie': 14, 'History': 15, 'War': 16, 'Adventure': 17, 'Music': 18, 'Action': 19}
Shape of train_movies_df 152000
Shape of test_movies_df 38000
Data of test_movies_df         genres_index                          keywords_embedding_string  \
139354            14   2.11839452e-01 -4.63067479e-02 -1.01120341e+0...   
162359            17   3.08757335e-01  1.64751038e-01 -3.72421890e-0...   

       genres_indices     id  \
139354            [8]  19083   
162359     [1, 10, 7]   6028   

                          original_title_embedding_string  \
139354  -3.05360734e-01 -1.89008236e-01 -4.38988447e-0...   
162359  -2.79812217e-01  9.85054374e-02 -2.32860893e-0...   

                                overview_embedding_string release_date  
139354  1.06348976e-01 -1.50255859e-04 

In [ ]:
print('Data Preparation Task Completed Sucessfully !!!!')

Data Preparation Task Completed Sucessfully !!!!
